# langchain_ollama smoke test

Tests the primary API surface of `langchain_ollama` without requiring a running Ollama server.
We only exercise constructors, class metadata, and offline-safe operations.

In [ ]:
# Public API surface: the top-level exports should be importable and present.
from langchain_ollama import ChatOllama, OllamaEmbeddings, OllamaLLM

for cls in (ChatOllama, OllamaEmbeddings, OllamaLLM):
    print(f"{cls.__name__}: module={cls.__module__}")
    assert isinstance(cls.__name__, str) and cls.__name__

# All three should be present on the package namespace
import langchain_ollama
for name in ('ChatOllama', 'OllamaEmbeddings', 'OllamaLLM'):
    assert hasattr(langchain_ollama, name), f"missing top-level export: {name}"
print("Top-level exports OK")

In [ ]:
# ChatOllama: construct without hitting the server, inspect config.
from langchain_ollama import ChatOllama
from langchain_core.language_models.chat_models import BaseChatModel

chat = ChatOllama(model="llama3.1", temperature=0.0, base_url="http://localhost:11434")

assert isinstance(chat, BaseChatModel)
assert chat.model == "llama3.1"
assert chat.temperature == 0.0
# _llm_type is a string identifier on chat models
print(f"_llm_type = {chat._llm_type}")
assert isinstance(chat._llm_type, str)
print("ChatOllama construction OK")

In [ ]:
# OllamaLLM: construct the completion-style LLM wrapper.
from langchain_ollama import OllamaLLM
from langchain_core.language_models.llms import BaseLLM

llm = OllamaLLM(model="llama3.1", temperature=0.2, num_predict=32)

assert isinstance(llm, BaseLLM)
assert llm.model == "llama3.1"
assert llm.temperature == 0.2
print(f"OllamaLLM._llm_type = {llm._llm_type}")
print(f"model={llm.model} num_predict={llm.num_predict}")
print("OllamaLLM construction OK")

In [ ]:
# OllamaEmbeddings: construct the embeddings wrapper.
from langchain_ollama import OllamaEmbeddings
from langchain_core.embeddings import Embeddings

emb = OllamaEmbeddings(model="nomic-embed-text", base_url="http://localhost:11434")

assert isinstance(emb, Embeddings)
assert emb.model == "nomic-embed-text"
# Both embed_query and embed_documents must be present
assert callable(getattr(emb, "embed_query", None))
assert callable(getattr(emb, "embed_documents", None))
print("OllamaEmbeddings construction OK")

In [ ]:
# Integration with langchain_core: ChatOllama works inside a prompt | model chain.
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are concise."),
    ("human", "{question}"),
])
chat = ChatOllama(model="llama3.1", temperature=0.0)
chain = prompt | chat

# Don't invoke (would require a running server); just confirm the pipe composed correctly.
from langchain_core.runnables import RunnableSequence
assert isinstance(chain, RunnableSequence)
rendered = prompt.invoke({"question": "What is 2+2?"})
assert rendered.to_messages()[-1].content == "What is 2+2?"
print(f"Chain type: {type(chain).__name__}")
print("langchain_ollama + langchain_core integration OK")